In [ ]:

import sys, os, glob, shutil, time, json, gc
import numpy as np, pandas as pd
t0 = time.perf_counter()
def log(m): print(f"[{time.perf_counter()-t0:6.0f}с] {m}", flush=True)

os.makedirs("/kaggle/working/src", exist_ok=True); os.makedirs("/kaggle/working/models", exist_ok=True)
code = os.path.dirname(glob.glob("/kaggle/input/**/pair_features.py", recursive=True)[0])
for p in glob.glob(code + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
for p in glob.glob(code + "/*.json"): shutil.copy(p, "/kaggle/working/models/")
struct = os.path.dirname(glob.glob("/kaggle/input/**/pair_boost_hybrid.npz", recursive=True)[0])
for p in glob.glob(struct + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
for p in glob.glob(struct + "/*.npz"): shutil.copy(p, "/kaggle/working/models/")
open("/kaggle/working/src/__init__.py", "a").close()
os.chdir("/kaggle/working"); sys.path.insert(0, "/kaggle/working")

from src.pair_features import build_matrix, feature_names
from src.features import extract_model_features
from src.model import BoostedPairModel
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
from scipy.stats import rankdata

fold = os.path.dirname(glob.glob("/kaggle/input/**/llm_valid_pairs.parquet", recursive=True)[0])
v2 = os.path.dirname(glob.glob("/kaggle/input/**/ce_balanced.npy", recursive=True)[0])
pairs = pd.read_parquet(fold + "/llm_valid_pairs.parquet")
items = pd.read_parquet(fold + "/llm_valid_items.parquet")
y = (pairs["target"].to_numpy() > 0).astype(np.int8)
cat = pairs["id1"].map(dict(zip(items.id, items.category.astype(str)))).astype(str).to_numpy()
known = sorted(set(items.category.astype(str)))

def align(scores_path, pairs_path, tag):
    """Скоры считались на 200 тысячах пар, фолд — 156 тысяч. Приводим по идентификаторам."""
    full = np.load(scores_path).astype(np.float32)
    src = pd.read_parquet(pairs_path)
    pos = pd.Series(np.arange(len(src)), index=pd.MultiIndex.from_arrays([src.id1, src.id2]))
    take = pos.reindex(pd.MultiIndex.from_arrays([pairs.id1, pairs.id2])).to_numpy()
    if not np.isfinite(take).all():
        raise SystemExit(f"{tag}: скоры не покрывают фолд")
    return full[take.astype(int)]

hn = glob.glob("/kaggle/input/**/ce_hardneg_scores.npy", recursive=True)[0]
dt = glob.glob("/kaggle/input/**/ce_bge/llm_ood_scores.npy", recursive=True)[0]
S = {"ce_spec": np.load(f"{v2}/ce_spec.npy").astype(np.float32),
     "ce_e5": np.load(f"{v2}/ce_e5.npy").astype(np.float32),
     "ce_hardneg": align(hn, os.path.dirname(hn) + "/ce_hardneg_pairs.parquet", "hardneg"),
     "ce_bge": align(dt, os.path.dirname(dt) + "/llm_ood_pairs.parquet", "bge")}
log(f"скоры собраны: {list(S)}")

names = list(feature_names(False, True))
X = np.zeros((len(pairs), len(names)), dtype=np.float32)
for c in known:
    rows = np.flatnonzero(cat == c)
    if not len(rows): continue
    sub = items[items.category.astype(str) == c].reset_index(drop=True)
    X[rows] = build_matrix(sub, pairs.iloc[rows].reset_index(drop=True), known,
                           with_neighbours=False, with_measures=True)
    del sub; gc.collect()
legacy = extract_model_features(pairs[["id1", "id2"]], items)
pr = BoostedPairModel("models/pair_boost_hybrid.npz").predict_probability(legacy, cat)
au = BoostedPairModel("models/pair_boost_hybrid_aux.npz").predict_probability(legacy, cat)
del legacy; gc.collect()
structural = 0.8 * pr + 0.2 * au
codes = np.array([known.index(c) if c in known else -1 for c in cat], dtype=np.float32)
log(f"признаки готовы: {X.shape}")

masks = {c: cat == c for c in np.unique(cat)}
def rk(s):
    o = np.empty(len(s), np.float32)
    for m in masks.values(): o[m] = rankdata(s[m]) / m.sum()
    return o
def macro(s, rate=0.111, seeds=8):
    vals = []
    for seed in range(seeds):
        rng = np.random.default_rng(seed); per = []
        for m in masks.values():
            rows = np.flatnonzero(m); pos_, neg = rows[y[rows] == 1], rows[y[rows] == 0]
            keep = min(len(pos_), max(5, int(round(rate / (1 - rate) * len(neg)))))
            ch = np.concatenate([rng.choice(pos_, keep, replace=False), neg])
            per.append(average_precision_score(y[ch], s[ch]))
        vals.append(np.mean(per))
    return float(np.mean(vals)), float(np.std(vals))

PARAMS = dict(max_iter=500, max_leaf_nodes=63, learning_rate=0.06, l2_regularization=1.0,
              early_stopping=False, random_state=0)
half = np.random.default_rng(5).permutation(len(y)) % 2
def run(combo, tag):
    mat = np.column_stack([X] + [S[e] for e in combo] + [structural, codes]).astype(np.float32)
    p = np.zeros(len(y))
    for h in (0, 1):
        tr, te = half != h, half == h
        p[te] = HistGradientBoostingClassifier(**PARAMS).fit(mat[tr], y[tr]).predict_proba(mat[te])[:, 1]
    mu, sd = macro(rk(p))
    log(f"  {tag:<34} {mu:.6f} ± {sd:.6f}")
    del mat; gc.collect()
    return mu

# Вопрос ровно один: заслуживает ли BGE места. Поэтому сначала наша тройка, потом она же
# плюс BGE, потом каждая замена члена тройки на BGE. Замена важнее добавления: четвёртый
# энкодер стоит времени на инференсе, а замена бесплатна.
log("=== тройка и что с ней делает BGE ===")
base = run(("ce_spec", "ce_e5", "ce_hardneg"), "тройка (наш кандидат)")
run(("ce_spec", "ce_e5", "ce_hardneg", "ce_bge"), "+ ce_bge четвёртой")
run(("ce_spec", "ce_e5", "ce_bge"), "ce_bge вместо ce_hardneg")
run(("ce_spec", "ce_hardneg", "ce_bge"), "ce_bge вместо ce_e5")
run(("ce_e5", "ce_hardneg", "ce_bge"), "ce_bge вместо ce_spec")
log(f"опора для сравнения: тройка {base:.6f}")
